# Pretraining

Cleaned, documented version of the supplied thesis notebook. Original training procedures, validation checks and numerical parameters are retained. Run cells in order. Configure input directories and checkpoint paths before execution.


## Environment and paths


In [ ]:
# ============================================================
# 0. IMPORTS AND CONFIGURATION
# ============================================================
import os
import csv
import json
import glob
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, random_split
from tqdm.auto import tqdm

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device:", device)
if device.type == "mps":
    print("Apple Silicon GPU activated via MPS/Metal")

# Pre-training dataset directory
PRETRAIN_DIR = "cartella_air14cm

VAL_FRACTION = 0.15
BATCH_SIZE = 256
NUM_WORKERS = 0

NUM_EPOCHS = 40

LR = 1e-3
WEIGHT_DECAY = 1e-5

ALPHA_MA = 0.01
BETA_THICK = 1.0

METRICS_EVERY = 10
PLOT_EVERY = 10

#checkpoint, best model and loss log path
BEST_MODEL_PATH = "best_model_pretrain_T10_FULL_all.pth"
CHECKPOINT_PATH = "best_checkpoint_pretrain_T10_FULL_all.pth"
CSV_LOG_PATH = "train_val_pretrain_T10_FULL_all.csv"

# False = start pre-training from scratch.
# True = resume from checkpoint.
RESUME = False
PRINT_EVERY = 10

## Dataset implementation


In [ ]:
# ============================================================
# DATASET cache optimization
# ============================================================
import os
import json
import numpy as np
import torch
from torch.utils.data import Dataset, random_split
from pathlib import Path
from tqdm.auto import tqdm


def _infer_elements_from_json(file_paths):
    elems = set()
    for path in tqdm(file_paths, desc="Infer elements", unit="file"):
        with open(path, "r") as f:
            d = json.load(f)
        for layer_comp in d["labels"]["composition"].values():
            elems.update(layer_comp.keys())
    return sorted(elems)


def build_binary_xrf_cache(
    file_paths,
    cache_dir,
    elements=None,
    add_org=True,
    normalize="max",
):
    """build_binary_xrf_cache: perform the processing step described by the implementation. See function arguments and returned values for details."""
    file_paths = sorted(map(str, file_paths))
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)

    meta_path = cache_dir / "meta.json"
    x_path = cache_dir / "X.npy"
    lp_path = cache_dir / "layer_present.npy"
    th_path = cache_dir / "thickness.npy"
    comp_path = cache_dir / "comp_full.npy"

    # check for existing valid cache
    if all(p.exists() for p in (meta_path, x_path, lp_path, th_path, comp_path)):
        meta = json.loads(meta_path.read_text())
        if meta.get("n_samples") == len(file_paths):
            print(f"Binary cache found: {cache_dir}")
            print(f"Cached samples: {meta['n_samples']}")
            return meta["elements"]

    if elements is None:
        elements = _infer_elements_from_json(file_paths)

    elem_to_idx = {e: i for i, e in enumerate(elements)}
    n = len(file_paths)
    n_layers = 3
    n_elems = len(elements) + (1 if add_org else 0)

    # Implementation note for this processing step.
    X = np.lib.format.open_memmap(
        x_path, mode="w+", dtype=np.float32, shape=(n, 1, 2048)
    )
    LP = np.lib.format.open_memmap(
        lp_path, mode="w+", dtype=np.float32, shape=(n, 3)
    )
    TH = np.lib.format.open_memmap(
        th_path, mode="w+", dtype=np.float32, shape=(n, 3)
    )
    COMP = np.lib.format.open_memmap(
        comp_path, mode="w+", dtype=np.float32, shape=(n, 3, n_elems)
    )

    org_idx = len(elements) if add_org else None

    for i, path in enumerate(tqdm(file_paths, desc="JSON -> binary cache", unit="file")):
        with open(path, "r") as f:
            d = json.load(f)

        counts = np.asarray(d["counts"], dtype=np.float32)

        if normalize == "max":
            m = float(counts.max())
            if m > 0:
                counts = counts / m
        elif normalize == "sum":
            ss = float(counts.sum())
            if ss > 0:
                counts = counts / ss
        elif normalize is not None:
            raise ValueError(f"normalize='{normalize}' not supported")

        labels = d["labels"]
        lp = np.asarray(labels["layer_present"], dtype=np.float32)
        th = np.asarray(labels["thickness_rho_t"], dtype=np.float32)
        organic = np.asarray(labels["organic_frac"], dtype=np.float32)

        comp = np.zeros((3, n_elems), dtype=np.float32)

        for li in range(3):
            lname = f"layer_{li+1}"
            for el, val in labels["composition"].get(lname, {}).items():
                j = elem_to_idx.get(el)
                if j is not None:
                    comp[li, j] = float(val)

            if add_org:
                comp[li, org_idx] = float(organic[li])

            if lp[li] > 0.5:
                total = float(comp[li].sum())
                if total > 0:
                    comp[li] /= total
            else:
                comp[li].fill(0.0)
                if add_org:
                    comp[li, org_idx] = 1.0

        X[i, 0] = counts
        LP[i] = lp
        TH[i] = th
        COMP[i] = comp

    # forza flush su disco
    X.flush(); LP.flush(); TH.flush(); COMP.flush()
    del X, LP, TH, COMP

    meta = {
        "n_samples": n,
        "elements": list(elements),
        "add_org": bool(add_org),
        "normalize": normalize,
        "source_first": file_paths[0] if file_paths else None,
        "source_last": file_paths[-1] if file_paths else None,
    }
    meta_path.write_text(json.dumps(meta, indent=2))

    print(f"Cache created: {cache_dir}")
    return list(elements)


class XRFBinaryDataset(Dataset):
    """XRFBinaryDataset: perform the processing step described by the implementation. See function arguments and returned values for details."""
    def __init__(self, cache_dir):
        self.cache_dir = Path(cache_dir)
        self.meta = json.loads((self.cache_dir / "meta.json").read_text())
        self.elements = self.meta["elements"]
        self.n = int(self.meta["n_samples"])

        self._X = None
        self._LP = None
        self._TH = None
        self._COMP = None

    def _open(self):
        if self._X is None:
            self._X = np.load(self.cache_dir / "X.npy", mmap_mode="r")
            self._LP = np.load(self.cache_dir / "layer_present.npy", mmap_mode="r")
            self._TH = np.load(self.cache_dir / "thickness.npy", mmap_mode="r")
            self._COMP = np.load(self.cache_dir / "comp_full.npy", mmap_mode="r")

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        self._open()

        # copy() evita warning su memmap read-only -> torch tensor
        x = torch.from_numpy(np.array(self._X[idx], copy=True))
        lp = torch.from_numpy(np.array(self._LP[idx], copy=True))
        th = torch.from_numpy(np.array(self._TH[idx], copy=True))
        comp = torch.from_numpy(np.array(self._COMP[idx], copy=True))

        return x, lp, th, comp


def split_xrf_dataset(dataset, val_fraction=0.2, seed=42):
    n_total = len(dataset)
    n_val = int(n_total * val_fraction)
    n_train = n_total - n_val
    generator = torch.Generator().manual_seed(seed)
    return random_split(dataset, [n_train, n_val], generator=generator)


## Data loaders


In [ ]:
# ============================================================
# 2. Pre-training dataloader construction
# ============================================================

json_files = sorted(str(p) for p in Path(PRETRAIN_DIR).glob("*.json"))

if len(json_files) == 0:
    raise FileNotFoundError(
        f"Nessun JSON trovato in '{PRETRAIN_DIR}'. Controlla PRETRAIN_DIR."
    )

#cache directory
CACHE_DIR = Path(f"{PRETRAIN_DIR}__binary_cache_max")

elements = build_binary_xrf_cache(
    file_paths=json_files,
    cache_dir=CACHE_DIR,
    elements=None,
    add_org=True,
    normalize="max",
)

dataset = XRFBinaryDataset(CACHE_DIR)

print("Directory:", PRETRAIN_DIR)
print("Cache:", CACHE_DIR)
print("JSON totali:", len(dataset))
print("Inorganic elements:", elements)
print("Number of inorganic elements:", len(elements))
print("N output composizione incl. Org:", len(elements) + 1)

train_dataset, val_dataset = split_xrf_dataset(
    dataset,
    val_fraction=VAL_FRACTION,
    seed=SEED,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

x0, lp0, th0, comp0 = next(iter(train_loader))

print("\nBatch check:")
print("x:", tuple(x0.shape))
print("layer_present:", tuple(lp0.shape))
print("thickness:", tuple(th0.shape))
print("comp_full:", tuple(comp0.shape))


## Network architecture — `XRFMultilayerNetLocalComp`

The input is a **2048-channel XRF spectrum** represented as `[batch, 1, 2048]`.

1. **Shared 1D convolutional encoder:** four Conv1d blocks with channels `1 → 64 → 128 → 256 → 256`, kernel size 7, stride 2, padding 3, ReLU and convolutional dropout (0.05).
2. **Presence head:** predicts the logit for the presence of the second paint layer (`presence2_logit`, shape `[B]`).
3. **Thickness head:** predicts `log10(ρt)` for three stratigraphic positions (`thickness_log`, shape `[B, 3]`).
4. **Local composition head:** processes local spectral features with convolution, adaptive pooling (8 bins), and a shared projection; predicts per-layer elemental composition logits, including the organic/other category (`comp_logits`, shape `[B, 3, M]`, where `M = len(elements) + 1`).

The three outputs are trained jointly. The architecture is identical in pre-training and fine-tuning; the latter initializes its parameters from the pre-trained checkpoint.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class XRFMultilayerNetLocalComp(nn.Module):
    """
    T10 FULL architecture.

    Encoder HPO fixed:
        1 -> 64 -> 128 -> 256 -> 256
        kernel=7, stride=2

    Local composition branch T10:
        pool=8
        local_kernel=7
        local_channels=256 -> 128
        shared_hidden=256
        local_dropout=0.10

    Output:
        presence2_logit : [B]
        thickness_log   : [B, 3]
        comp_logits     : [B, 3, len(elements)+1]
    """
    def __init__(
        self,
        n_channels=1,
        spec_len=2048,
        elements=None,
        latent_dim=256,
        comp_pool_bins=8,
        comp_hidden=256,
        local_kernel=7,
        local_channels=256,
        local_dropout=0.10,
        drop_conv=0.05,
        drop_latent=0.10,
        drop_head=0.00,
    ):
        super().__init__()
        if elements is None:
            elements = []

        self.elements = list(elements)
        self.n_layers = 3
        self.n_elems = len(elements) + 1
        self.latent_dim = 256
        self.comp_pool_bins = int(comp_pool_bins)
        self.local_kernel = int(local_kernel)
        self.local_channels = int(local_channels)
        self.local_dropout = float(local_dropout)

        self.do_conv = nn.Dropout1d(p=drop_conv) if drop_conv > 0 else nn.Identity()
        self.do_latent = nn.Dropout(p=drop_latent) if drop_latent > 0 else nn.Identity()
        self.do_head = nn.Dropout(p=drop_head) if drop_head > 0 else nn.Identity()

        self.encoder = nn.Sequential(
            nn.Conv1d(n_channels, 64, kernel_size=7, stride=2, padding=3),
            nn.ReLU(inplace=True),
            self.do_conv,

            nn.Conv1d(64, 128, kernel_size=7, stride=2, padding=3),
            nn.ReLU(inplace=True),
            self.do_conv,

            nn.Conv1d(128, 256, kernel_size=7, stride=2, padding=3),
            nn.ReLU(inplace=True),
            self.do_conv,

            nn.Conv1d(256, 256, kernel_size=7, stride=2, padding=3),
            nn.ReLU(inplace=True),
            self.do_conv,
        )

        self.head_presence2 = nn.Linear(256, 1)
        self.head_thickness_log = nn.Linear(256, self.n_layers)

        pad_local = self.local_kernel // 2
        local_hidden_2 = max(self.local_channels // 2, 1)

        comp_pre_layers = [
            nn.Conv1d(256, self.local_channels, kernel_size=self.local_kernel, padding=pad_local),
            nn.ReLU(inplace=True),
        ]
        if self.local_dropout > 0:
            comp_pre_layers.append(nn.Dropout1d(p=self.local_dropout))

        comp_pre_layers += [
            nn.Conv1d(self.local_channels, local_hidden_2, kernel_size=self.local_kernel, padding=pad_local),
            nn.ReLU(inplace=True),
        ]
        if self.local_dropout > 0:
            comp_pre_layers.append(nn.Dropout1d(p=self.local_dropout))

        self.comp_pre = nn.Sequential(*comp_pre_layers)
        self.comp_pool = nn.AdaptiveAvgPool1d(self.comp_pool_bins)

        comp_in_dim = local_hidden_2 * self.comp_pool_bins + 256
        self.comp_shared = nn.Sequential(
            nn.Linear(comp_in_dim, comp_hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(p=drop_head) if drop_head > 0 else nn.Identity(),
        )

        self.head_comp_l1 = nn.Linear(comp_hidden, self.n_elems)
        self.head_comp_l2 = nn.Linear(comp_hidden, self.n_elems)
        self.head_comp_l3 = nn.Linear(comp_hidden, self.n_elems)

    def forward(self, x):
        z_map = self.encoder(x)
        z_global = self.do_latent(z_map.mean(dim=-1))
        z_head = self.do_head(z_global)

        presence2_logit = self.head_presence2(z_head).squeeze(-1)
        thickness_log = self.head_thickness_log(z_head)

        z_comp_local = self.comp_pre(z_map)
        z_comp_local = self.comp_pool(z_comp_local).flatten(1)
        z_comp = torch.cat([z_comp_local, z_global], dim=1)
        z_comp = self.comp_shared(z_comp)

        comp_logits = torch.stack([
            self.head_comp_l1(z_comp),
            self.head_comp_l2(z_comp),
            self.head_comp_l3(z_comp),
        ], dim=1)

        return {
            "z_global": z_global,
            "z_map": z_map,
            "presence2_logit": presence2_logit,
            "thickness_log": thickness_log,
            "comp_logits": comp_logits,
        }


## Pre-training step


In [ ]:
# ============================================================
# 3. Model definition and initialization
# ============================================================
model = XRFMultilayerNetLocalComp(
    n_channels=1,
    spec_len=2048,
    elements=elements,
    latent_dim=256,
    comp_pool_bins=8,
    comp_hidden=256,
    local_kernel=7,
    local_channels=256,
    local_dropout=0.10,
    drop_conv=0.05,
    drop_latent=0.10,
    drop_head=0.00,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("T10 FULL model")
print(f"Parametri totali: {n_params:,}")
print(f"Parametri trainabili: {n_trainable:,}")



## Multi-task loss — three original objectives

Let $B$ be the batch size, $L=3$ the number of stratigraphic positions, $M$ the number of composition categories, $m_{b,l}\in\{0,1\}$ the layer-presence mask, and $N_{\mathrm{valid}}=\sum_{b,l}m_{b,l}$.

**1. Layer-presence loss (binary cross-entropy with logits)**, applied to the **second layer only**:

$$
\mathcal{L}_{\mathrm{presence}} =
-\frac{1}{B}\sum_{b=1}^{B}
\left[
y_{b,2}\log\sigma(z_b)
+(1-y_{b,2})\log(1-\sigma(z_b))
\right].
$$

**2. Thickness loss (masked mean squared error)**, using the logarithm of the *areal density* $\rho t$ for present layers:

$$
\mathcal{L}_{\mathrm{thickness}} =
\frac{1}{N_{\mathrm{valid}}}
\sum_{b=1}^{B}\sum_{l=1}^{L}m_{b,l}
\left[
\widehat{\log_{10}(\rho t)}_{b,l}
-\log_{10}((\rho t)_{b,l})
\right]^2.
$$

**3. Composition loss (masked soft-label cross-entropy)**:

$$
\mathcal{L}_{\mathrm{composition}} =
-\frac{1}{N_{\mathrm{valid}}+\varepsilon}
\sum_{b=1}^{B}\sum_{l=1}^{L}m_{b,l}
\sum_{j=1}^{M}c_{b,l,j}
\log\left[
\operatorname{softmax}(z^{(c)}_{b,l})_j
\right].
$$

The function below returns these three **raw** losses separately. The training loop combines them using the original moving-average normalization and thickness weighting; it is **not** a simple unweighted sum.



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def xrf_multilayer_loss_raw(outputs,
                            layer_present,
                            thickness,
                            comp_true,
                            eps=1e-8):
    """Return the three raw multi-task losses: binary presence BCE for layer 2, masked MSE of log10 areal density, and masked soft-label composition cross-entropy."""
    presence2_logit = outputs["presence2_logit"]      # [B]
    thickness_log_pred = outputs["thickness_log"]     # [B, 3]
    comp_logits = outputs["comp_logits"]              # [B, 3, M]

    device = presence2_logit.device

    # 1) Presence (BCE) sul layer_2
    target_layer2 = layer_present[:, 1]  # [B]
    bce = nn.BCEWithLogitsLoss()
    loss_presence = bce(presence2_logit, target_layer2)

    # Layer-wise processing and masking.
    mask = (layer_present > 0.5)  # [B,3] bool
    valid_thickness = thickness[mask]                  # [N_valid]
    valid_thickness_log_pred = thickness_log_pred[mask]  # [N_valid]

    if valid_thickness.numel() > 0:
        valid_thickness_log_true = torch.log10(valid_thickness)
        loss_thickness = F.mse_loss(valid_thickness_log_pred,
                                    valid_thickness_log_true)
    else:
        loss_thickness = torch.tensor(0.0, device=device)

    # Layer-wise processing and masking.
    log_probs = F.log_softmax(comp_logits, dim=-1)       # [B,3,M]
    ce_per_layer = -(comp_true * log_probs).sum(dim=-1)  # [B,3]

    ce_masked = ce_per_layer * mask.float()
    if mask.sum() > 0:
        loss_comp = ce_masked.sum() / (mask.sum() + eps)
    else:
        loss_comp = torch.tensor(0.0, device=device)

    return loss_presence, loss_thickness, loss_comp

## Pre-training step


In [ ]:
def train_one_epoch(
    model,
    train_loader,
    optimizer,
    device,
    epoch,
    ma_pres,
    ma_thick,
    ma_comp,
    alpha_ma,
    beta_thick,
):
    model.train()

    running_loss_norm = 0.0
    running_loss_raw = 0.0
    running_pres = 0.0
    running_thick = 0.0
    running_comp = 0.0
    n_batches = 0

    pbar = tqdm(
        train_loader,
        desc=f"Train {epoch:03d}",
        leave=True,
        dynamic_ncols=True,
    )

    for batch in pbar:

        x, layer_present, thickness, comp_full = batch

        x = x.to(device)
        layer_present = layer_present.to(device)
        thickness = thickness.to(device)
        comp_full = comp_full.to(device)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(x)

        loss_pres, loss_thick, loss_comp = xrf_multilayer_loss_raw(
            outputs,
            layer_present,
            thickness,
            comp_full,
        )

        # ----------------------------------------------------
        # Moving averages
        # ----------------------------------------------------
        with torch.no_grad():
            ma_pres = (
                (1.0 - alpha_ma) * ma_pres
                + alpha_ma * loss_pres.item()
            )

            ma_thick = (
                (1.0 - alpha_ma) * ma_thick
                + alpha_ma * loss_thick.item()
            )

            ma_comp = (
                (1.0 - alpha_ma) * ma_comp
                + alpha_ma * loss_comp.item()
            )

        # ----------------------------------------------------
        # Normalized loss
        # ----------------------------------------------------
        loss_norm = (
            loss_pres / (ma_pres + 1e-8)
            + beta_thick * loss_thick / (ma_thick + 1e-8)
            + loss_comp / (ma_comp + 1e-8)
        )

        loss_raw = (
            loss_pres
            + beta_thick * loss_thick
            + loss_comp
        )

        loss_norm.backward()
        optimizer.step()

        running_loss_norm += loss_norm.item()
        running_loss_raw += loss_raw.item()

        running_pres += loss_pres.item()
        running_thick += loss_thick.item()
        running_comp += loss_comp.item()

        n_batches += 1

        pbar.set_postfix(
            loss=f"{running_loss_norm / n_batches:.4f}",
            pres=f"{running_pres / n_batches:.4f}",
            thick=f"{running_thick / n_batches:.4f}",
            comp=f"{running_comp / n_batches:.4f}",
        )

    metrics = {
        "loss_norm": running_loss_norm / n_batches,
        "loss_raw": running_loss_raw / n_batches,
        "presence": running_pres / n_batches,
        "thickness": running_thick / n_batches,
        "composition": running_comp / n_batches,
    }

    return metrics, ma_pres, ma_thick, ma_comp

## Pre-training step


In [ ]:
@torch.no_grad()
def validate_one_epoch(
    model,
    val_loader,
    device,
    epoch,
):
    model.eval()

    running_loss_raw = 0.0
    running_pres = 0.0
    running_thick = 0.0
    running_comp = 0.0
    n_batches = 0

    pbar = tqdm(
        val_loader,
        desc=f"Val   {epoch:03d}",
        leave=True,
        dynamic_ncols=True,
    )

    for batch in pbar:

        x, layer_present, thickness, comp_full = batch

        x = x.to(device)
        layer_present = layer_present.to(device)
        thickness = thickness.to(device)
        comp_full = comp_full.to(device)

        outputs = model(x)

        loss_pres, loss_thick, loss_comp = xrf_multilayer_loss_raw(
            outputs,
            layer_present,
            thickness,
            comp_full,
        )

        loss_raw = (
            loss_pres
            + loss_thick
            + loss_comp
        )

        running_loss_raw += loss_raw.item()

        running_pres += loss_pres.item()
        running_thick += loss_thick.item()
        running_comp += loss_comp.item()

        n_batches += 1

        pbar.set_postfix(
            loss=f"{running_loss_raw / n_batches:.4f}",
            pres=f"{running_pres / n_batches:.4f}",
            thick=f"{running_thick / n_batches:.4f}",
            comp=f"{running_comp / n_batches:.4f}",
        )

    return {
        "loss_raw": running_loss_raw / n_batches,
        "presence": running_pres / n_batches,
        "thickness": running_thick / n_batches,
        "composition": running_comp / n_batches,
    }

## Evaluation metrics on the validation set


In [ ]:
#L2-detection accuracy
@torch.no_grad()
def eval_presence_accuracy(model, data_loader, device, threshold=0.5):
    model.eval()
    correct = 0
    total = 0
    for x, layer_present, thickness, comp_full in data_loader:
        x = x.to(device)
        layer_present = layer_present.to(device)
        outputs = model(x)
        probs = torch.sigmoid(outputs["presence2_logit"])  # [B]
        preds = (probs > threshold).float()
        target = layer_present[:, 1]  # Layer-wise processing and masking.
        correct += (preds == target).float().sum().item()
        total += target.numel()
    return correct / total

#Areal density MAE
@torch.no_grad()
def eval_thickness_mae(model, data_loader, device):
    model.eval()
    abs_errors = []
    for x, layer_present, thickness, comp_full in data_loader:
        x = x.to(device)
        layer_present = layer_present.to(device)
        thickness = thickness.to(device)

        outputs = model(x)
        th_log_pred = outputs["thickness_log"]  # [B,3]
        th_pred = 10.0 ** th_log_pred           # [B,3]

        mask = (layer_present > 0.5)
        if mask.any():
            err = torch.abs(th_pred - thickness)[mask]
            abs_errors.append(err.detach().cpu())

    if len(abs_errors) == 0:
        return None
    abs_errors = torch.cat(abs_errors, dim=0)
    return abs_errors.mean().item(), abs_errors.median().item()

#areal density relative error
@torch.no_grad()
def eval_thickness_relative_error(model, data_loader, device, eps=1e-12):
    model.eval()
    rel_errors = []

    for x, layer_present, thickness, comp_full in data_loader:
        x = x.to(device)
        layer_present = layer_present.to(device)
        thickness = thickness.to(device)

        outputs = model(x)
        t_log_pred = outputs["thickness_log"]  # [B,3]
        t_pred = 10 ** t_log_pred              # [B,3]

        mask = (layer_present > 0.5)

        if mask.any():
            # Relative error: |pred - true| / true
            rel = torch.abs(t_pred - thickness) / (thickness + eps)
            rel = rel[mask]
            rel_errors.append(rel.detach().cpu())

    if len(rel_errors) == 0:
        return None, None

    rel_errors = torch.cat(rel_errors, dim=0)
    return rel_errors.mean().item(), rel_errors.median().item()


import numpy as np
import torch
import torch.nn.functional as F

#Composition metrics 
@torch.no_grad()
def eval_composition_metrics(
    model,
    data_loader,
    device,
    org_index=None,
    org_min_true=1e-3,
    topk=(1, 3),
):
    """eval_composition_metrics: perform the processing step described by the implementation. See function arguments and returned values for details."""

    model.eval()

    l1_list = []
    l2_list = []
    kl_list = []

    top1_correct = 0
    top3_correct = 0
    total_layers = 0

    org_abs_errors = []
    org_rel_errors = []

    for x, layer_present, thickness, comp_full in data_loader:
        x = x.to(device)
        layer_present = layer_present.to(device)
        comp_full = comp_full.to(device)

        outputs = model(x)
        comp_logits = outputs["comp_logits"]          # [B, 3, M]
        comp_pred = F.softmax(comp_logits, dim=-1)    # [B, 3, M]

        # Layer-wise processing and masking.
        mask = (layer_present > 0.5)                  # [B, 3]
        if mask.sum() == 0:
            continue

        # Layer-wise processing and masking.
        true_flat = comp_full[mask]                   # [N_layers, M]
        pred_flat = comp_pred[mask]                   # [N_layers, M]

       
        true_flat = true_flat / (true_flat.sum(dim=-1, keepdim=True) + 1e-8)
        pred_flat = pred_flat / (pred_flat.sum(dim=-1, keepdim=True) + 1e-8)

        # Layer-wise processing and masking.
        l1 = torch.sum(torch.abs(pred_flat - true_flat), dim=-1)          # [N_layers]
        l2 = torch.sqrt(torch.sum((pred_flat - true_flat) ** 2, dim=-1))  # [N_layers]

        l1_list.append(l1.cpu().numpy())
        l2_list.append(l2.cpu().numpy())

      
        eps = 1e-8
        mask_pos = (true_flat > 0)
        kl = torch.sum(
            torch.where(
                mask_pos,
                true_flat * (torch.log(true_flat + eps) - torch.log(pred_flat + eps)),
                torch.zeros_like(true_flat),
            ),
            dim=-1,
        )  # [N_layers]
        kl_list.append(kl.cpu().numpy())

        # --------------------------------------------------------
        # Top-1 and Top-3 for dominant elements
        # Implementation note for this processing step.
        # --------------------------------------------------------
        if org_index is not None:
            # Dataset preparation.
            true_inorg = true_flat[:, :org_index]   # [N_layers, M_inorg]
            pred_inorg = pred_flat[:, :org_index]

            true_top1 = torch.argmax(true_inorg, dim=-1)   # [N_layers]
            pred_top1 = torch.argmax(pred_inorg, dim=-1)   # [N_layers]
        else:
            # Implementation note for this processing step.
            true_top1 = torch.argmax(true_flat, dim=-1)
            pred_top1 = torch.argmax(pred_flat, dim=-1)

        top1_correct += (true_top1 == pred_top1).sum().item()

        # top-3 overlap
        if 3 in topk:
            if org_index is not None:
                k = min(3, org_index)  
                true_topk = torch.topk(true_inorg, k=k, dim=-1).indices  # [N_layers, k]
                pred_topk = torch.topk(pred_inorg, k=k, dim=-1).indices  # [N_layers, k]
            else:
                k = 3
                true_topk = torch.topk(true_flat, k=k, dim=-1).indices
                pred_topk = torch.topk(pred_flat, k=k, dim=-1).indices

            # Layer-wise processing and masking.
            for i in range(true_flat.size(0)):
                set_true = set(true_topk[i].tolist())
                set_pred = set(pred_topk[i].tolist())
                if len(set_true.intersection(set_pred)) > 0:
                    top3_correct += 1

        total_layers += true_flat.size(0)

    
        if org_index is not None:
            org_true = true_flat[:, org_index]  # [N_layers]
            org_pred = pred_flat[:, org_index]

            abs_err = torch.abs(org_pred - org_true)
            org_abs_errors.append(abs_err.cpu().numpy())

         
            mask_org = (org_true > org_min_true)
            if mask_org.any():
                rel_err = (abs_err[mask_org] / org_true[mask_org])
                org_rel_errors.append(rel_err.cpu().numpy())

    if total_layers == 0:
        return {}

    l1_all = np.concatenate(l1_list, axis=0)
    l2_all = np.concatenate(l2_list, axis=0)
    kl_all = np.concatenate(kl_list, axis=0)

    metrics = {
        "num_layers": int(total_layers),
        "l1_mean": float(l1_all.mean()),
        "l1_median": float(np.median(l1_all)),
        "l2_mean": float(l2_all.mean()),
        "l2_median": float(np.median(l2_all)),
        "kl_mean": float(kl_all.mean()),
        "top1_acc": top1_correct / total_layers,
        "top3_acc": top3_correct / total_layers if total_layers > 0 else 0.0,
    }

    if org_index is not None and len(org_abs_errors) > 0:
        org_abs_all = np.concatenate(org_abs_errors, axis=0)
        metrics["org_mae"] = float(org_abs_all.mean())

        if len(org_rel_errors) > 0:
            org_rel_all = np.concatenate(org_rel_errors, axis=0)
            metrics["org_rel_mean"] = float(org_rel_all.mean())
            metrics["org_rel_median"] = float(np.median(org_rel_all))

    return metrics

## Composition report


In [ ]:
def print_composition_report(metrics):
    """print_composition_report: perform the processing step described by the implementation. See function arguments and returned values for details."""

    if not metrics:
        print("No layer evaluated.")
        return

    print("=== Composition metrics ===")
    print(f"- Num. layer valutati      : {metrics['num_layers']}")

    print("\n> Distribution errors (all elements, including Org)")
    print(f"  L1 mean                  : {metrics['l1_mean']:.4f} "
          f"(sum |pred-true| mean per layer)")
    print(f"  L1 median                : {metrics['l1_median']:.4f} "
          f"(median)")
    print(f"  L2 mean                  : {metrics['l2_mean']:.4f} "
          f"(RMSE over composition per layer)")
    print(f"  L2 median                : {metrics['l2_median']:.4f}")
    print(f"  KL mean (true || pred)   : {metrics['kl_mean']:.4f} ")

    print("\n> Dominant elements")
    print(f"  Top-1 accuracy           : {metrics['top1_acc']:.3f} ")
    print(f"  Top-3 accuracy           : {metrics['top3_acc']:.3f} ")

    if "org_mae" in metrics:
        print("\n> Organic component (Org)")
        print(f"  Org MAE                  : {metrics['org_mae']:.4f} "
              f"(mean absolute error in the organic fraction)")
        if "org_rel_mean" in metrics:
            print(f"  Org REL mean             : {metrics['org_rel_mean']:.3f} "
                  f"(mean relative error for Org, only where Org_true>{1e-3})")
            print(f"  Org REL median           : {metrics['org_rel_median']:.3f} "
                  f"(median relative error for Org)")


import matplotlib.pyplot as plt
import numpy as np

def plot_loss_curves(train_losses, val_losses, title="Loss curves"):
    """
    Plot loss curve per epoch:
    """
    train_losses = np.asarray(train_losses, dtype=float)
    val_losses = np.asarray(val_losses, dtype=float)

    epochs = np.arange(1, len(train_losses) + 1)

    plt.figure(figsize=(7, 5))
    plt.plot(epochs, train_losses, marker="o", label="Train loss (norm)")
    plt.plot(epochs, val_losses, marker="o", label="Val loss (raw)")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

## val and train loss save


In [ ]:
import os
import csv

CSV_LOG_PATH = "pre_train_T10.csv"

def log_epoch_to_csv(csv_path, epoch, train_metrics, val_metrics):
    """log_epoch_to_csv: perform the processing step described by the implementation. See function arguments and returned values for details."""
    file_exists = os.path.exists(csv_path)

    with open(csv_path, mode="a", newline="") as f:
        writer = csv.writer(f)

        if not file_exists:
            writer.writerow([
                "epoch",
                "train_loss_raw",
                "train_loss_norm",
                "train_loss_pres",
                "train_loss_thick",
                "train_loss_comp",
                "val_loss_raw",
                "val_loss_pres",
                "val_loss_thick",
                "val_loss_comp",
            ])

        writer.writerow([
            epoch,
            train_metrics["loss_raw"],
            train_metrics["loss_norm"],
            train_metrics["presence"],
            train_metrics["thickness"],
            train_metrics["composition"],
            val_metrics["loss_raw"],
            val_metrics["presence"],
            val_metrics["thickness"],
            val_metrics["composition"],
        ])

## Pre-training step


In [ ]:
# ============================================================
# 8. SETUP PRE-TRAINING
# ============================================================
start_epoch = 1
best_val_loss = float("inf")

#moving average weight initialization
ma_pres = 1.0
ma_thick = 1.0
ma_comp = 1.0

train_loss_history = []
val_loss_history = []

#checkpoint initialization (if RESUME=True)
if RESUME and os.path.exists(CHECKPOINT_PATH):
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)

    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])

    start_epoch = checkpoint["epoch"] + 1
    best_val_loss = checkpoint.get("best_val_loss", float("inf"))

    ma_pres = checkpoint.get("ma_pres", 1.0)
    ma_thick = checkpoint.get("ma_thick", 1.0)
    ma_comp = checkpoint.get("ma_comp", 1.0)

    train_loss_history = checkpoint.get("train_loss_history", [])
    val_loss_history = checkpoint.get("val_loss_history", [])

    print(
        f"RESUME from epoch {start_epoch} | "
        f"best_val_loss={best_val_loss:.6f}"
    )
else:
    print("Pre-training from scratch.")

print(
    f"Epochs: {start_epoch} -> {NUM_EPOCHS} | "
    f"alpha_ma={ALPHA_MA} | beta_thick={BETA_THICK}"
)

## Pre-training loop


In [ ]:
# ============================================================
# 9. PRE-TRAINING LOOP
# ============================================================
org_index = len(elements)

for epoch in range(start_epoch, NUM_EPOCHS + 1):

    # --------------------------------------------------------
    # Dropout behavior during training or evaluation.
    # --------------------------------------------------------
    train_metrics, ma_pres, ma_thick, ma_comp = train_one_epoch(
        model=model,
        train_loader=train_loader,
        optimizer=optimizer,
        device=device,
        epoch=epoch,
        ma_pres=ma_pres,
        ma_thick=ma_thick,
        ma_comp=ma_comp,
        alpha_ma=ALPHA_MA,
        beta_thick=BETA_THICK,
    )

    # --------------------------------------------------------
    # Validation and diagnostic metrics.
    # -> DROPOUT OFF
    # --------------------------------------------------------
    val_metrics = validate_one_epoch(
        model=model,
        val_loader=val_loader,
        device=device,
        epoch=epoch,
    )

    train_loss_history.append(train_metrics["loss_raw"])
    val_loss_history.append(val_metrics["loss_raw"])

    if epoch % PRINT_EVERY == 0 or epoch == NUM_EPOCHS:
        print(
            f"\n[Epoch {epoch:03d}] "
            f"Train: loss(norm)={train_metrics['loss_norm']:.4f} "
            f"| raw={train_metrics['loss_raw']:.4f} "
            f"(pres={train_metrics['presence']:.4f}, "
            f"thick={train_metrics['thickness']:.4f}, "
            f"comp={train_metrics['composition']:.4f}) "
            f"| Val(raw)={val_metrics['loss_raw']:.4f} "
            f"(pres={val_metrics['presence']:.4f}, "
            f"thick={val_metrics['thickness']:.4f}, "
            f"comp={val_metrics['composition']:.4f})"
        )

        print(
            f"    Moving averages: "
            f"pres={ma_pres:.5f} | "
            f"thick={ma_thick:.5f} | "
            f"comp={ma_comp:.5f}"
        )

    # CSV
    log_epoch_to_csv(
        CSV_LOG_PATH,
        epoch,
        train_metrics,
        val_metrics,
    )

    # --------------------------------------------------------
    # BEST MODEL: deterministic validation raw 
    # --------------------------------------------------------
    val_loss = val_metrics["loss_raw"]

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            BEST_MODEL_PATH,
        )

        if epoch % PRINT_EVERY == 0 or epoch == NUM_EPOCHS:
            print(
                f"    -> NEW BEST @ epoch {epoch}: "
                f"val_loss={best_val_loss:.6f}"
            )

    # --------------------------------------------------------
    # Checkpoint management and model state.
    # --------------------------------------------------------
    torch.save(
        {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "best_val_loss": best_val_loss,

            "ma_pres": ma_pres,
            "ma_thick": ma_thick,
            "ma_comp": ma_comp,

            "train_loss_history": train_loss_history,
            "val_loss_history": val_loss_history,

            "elements": elements,
            "pretrain_dir": PRETRAIN_DIR,
        },
        CHECKPOINT_PATH,
    )

    # --------------------------------------------------------
    # Loss curve plot
    # --------------------------------------------------------
    if epoch % PLOT_EVERY == 0:
        plot_loss_curves(
            train_loss_history,
            val_loss_history,
            title=f"Pre-training loss fino a epoch {epoch}",
        )


print("\nTraining ended.")
print("Best validation loss:", best_val_loss)
print("Best model:", BEST_MODEL_PATH)
print("Checkpoint:", CHECKPOINT_PATH)

plot_loss_curves(
    train_loss_history,
    val_loss_history,
    title="Pre-training loss history",
)

## Model final validation


In [ ]:
# ============================================================
# 10. BEST MODEL -> final validation
# ============================================================
best_model = XRFMultilayerNetLocalComp(
    n_channels=1,
    spec_len=2048,
    elements=elements,
    latent_dim=256,
    comp_pool_bins=8,
    comp_hidden=256,
    local_kernel=7,
    local_channels=256,
    local_dropout=0.10,
    drop_conv=0.05,
    drop_latent=0.10,
    drop_head=0.00,
).to(device)

best_model.load_state_dict(
    torch.load(BEST_MODEL_PATH, map_location=device)
)

final_val = validate_one_epoch(
    best_model,
    val_loader,
    device,
    epoch="BEST",
)

print("Final validation:")
print(final_val)

acc_pres = eval_presence_accuracy(best_model, val_loader, device)
mae_mean, mae_median = eval_thickness_mae(best_model, val_loader, device)
rel_mean, rel_median = eval_thickness_relative_error(best_model, val_loader, device)

metrics_comp = eval_composition_metrics(
    model=best_model,
    data_loader=val_loader,
    device=device,
    org_index=len(elements),
)

print(f"Presence accuracy: {acc_pres:.4f}")
print(f"Thickness MAE mean={mae_mean:.4e} median={mae_median:.4e}")
print(f"Thickness REL mean={rel_mean:.4f} median={rel_median:.4f}")
print_composition_report(metrics_comp)